In [3]:
!pip install ultralytics

  Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached torch-2.7.0-cp312-cp312-win_amd64.whl.metadata (29 kB)
  Using cached torchvision-0.22.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl.metadata (9.4 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 12.0 MB/s eta 0:00:00
Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl (39.5 MB)
Using cached torch-2.7.0-cp312-cp312-win_amd64.whl (212.5 MB)
Using cached torchvision-0.22.0-cp312-cp312-win_amd64.whl (1.7 MB)
Using cached ultralytics_thop-2.0.14-py3-none-any.whl (26 kB)
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ------------------- -------------------- 3.1/6.3 MB 14.1 MB/s eta 0:00:01
   ---------------------------------------  6.3/6.3 MB 15.4 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 13.8 MB/s et

In [1]:
from ultralytics import YOLO
import cv2
import tkinter as tk
from PIL import Image, ImageTk

# Load pre-trained YOLOv8 model
model = YOLO("yolov8n.pt")

# Total seats in the bus
TOTAL_SEATS = 30

# Setup Tkinter GUI
root = tk.Tk()
root.title("YOLOv8 Bus Occupancy Monitor")
root.geometry("800x700")  # Taller to fit more info

label = tk.Label(root)
label.pack()

# Person count and occupancy labels
count_label = tk.Label(root, text="Persons Detected: 0", font=("Arial", 16))
count_label.pack()

occupancy_label = tk.Label(root, text=f"Available Seats: {TOTAL_SEATS} / {TOTAL_SEATS}", font=("Arial", 16))
occupancy_label.pack()

cap = cv2.VideoCapture(0)

# Update function to display frames
def update_frame():
    ret, frame = cap.read()
    if not ret:
        return

    # Run YOLOv8 detection
    results = model(frame, verbose=False)

    # Filter detections to only 'person' (class_id 0)
    person_boxes = []
    for box in results[0].boxes:
        if int(box.cls[0]) == 0:  # Class 0 = person
            person_boxes.append(box)

    # Count detected persons
    person_count = len(person_boxes)
    available_seats = max(0, TOTAL_SEATS - person_count)

    # Update labels
    count_label.config(text=f"Persons Detected: {person_count}")
    occupancy_label.config(text=f"Available Seats: {available_seats} / {TOTAL_SEATS}")

    # Replace original boxes with person-only boxes
    results[0].boxes = person_boxes
    annotated_frame = results[0].plot()

    # Convert to RGB for Tkinter
    rgb_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb_frame)
    imgtk = ImageTk.PhotoImage(image=img)

    label.imgtk = imgtk
    label.configure(image=imgtk)

    # Refresh after 10ms
    root.after(10, update_frame)

# On close, release camera
def on_close():
    cap.release()
    root.destroy()

root.protocol("WM_DELETE_WINDOW", on_close)

update_frame()
root.mainloop()
